# Whale Clustering Experiment

Lean runner notebook for the end-to-end whale picture clustering pipeline described in `plan.md`. The notebook configures a run, calls helper modules, and displays summaries/review plots.

In [ ]:
from pathlib import Path
from pprint import pprint
import json
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from whale_clustering_helpers import (
    WhaleClusteringConfig,
    build_manifest,
    cluster_day,
    compute_artifacts,
    detect_series,
    evaluate_clusters,
    initialize_run,
    load_ground_truth,
    score_series_matches,
)
from plotting_utils import plot_cluster_review, plot_image_grid, plot_manifest_summary, plot_series_grid

print(f"Repo root: {REPO_ROOT}")

## 1. Load `plan.md` and Existing Pipeline State

Read the implementation plan and list existing run artifacts so the notebook can resume from cached previews, crops, embeddings, reports, or cluster exports.

In [ ]:
plan_text = (REPO_ROOT / "plan.md").read_text(encoding="utf-8")
plan_headings = [line for line in plan_text.splitlines() if line.startswith("##") or line.startswith("### Milestone")]
print("Plan headings and milestones:")
for line in plan_headings:
    print(line)

run_root = REPO_ROOT / "runs" / "whale_clustering"
existing_artifacts = sorted(path.relative_to(REPO_ROOT) for path in run_root.rglob("*") if path.is_file()) if run_root.exists() else []
print(f"\nExisting artifact files: {len(existing_artifacts)}")
for path in existing_artifacts[:25]:
    print(path)

## 2. Verify Dataset Paths and Metadata

Configure the run, inspect the Lightroom catalog, build the image manifest, and summarize image counts, label coverage, formats, days, and camera streams.

In [ ]:
config = WhaleClusteringConfig(
    image_root="/home/sat3737/Test/Lightroom images",
    lrcat_path="/home/sat3737/Test/Test.lrcat",
    run_dir="../runs/whale_clustering/baseline_zero_shot",
    target_day="2025-01-15",
    feature_method="color_texture",
)
run_dirs = initialize_run(config)
print(f"Run directory: {config.resolved_run_dir}")
print(f"Image root exists: {Path(config.image_root).expanduser().exists()}")
print(f"Lightroom catalog exists: {Path(config.lrcat_path).expanduser().exists()}")

ground_truth = load_ground_truth(config)
manifest = build_manifest(config, ground_truth)

print("\nGround truth summary:")
pprint(ground_truth["summary"])
print("\nManifest summary:")
pprint(manifest["summary"])
print("\nAvailable days in current manifest:")
pprint(manifest["available_days"][:20])

## 3. Inspect Whale Image Samples

Render a small set of manifest images. RAW files may show as unavailable until the preprocessing cell creates cached JPEG previews.

In [ ]:
plot_manifest_summary(manifest)
plot_image_grid(manifest, max_images=12)

## 4. Preprocess and Cache Images

Detect provisional temporal series, build JPEG previews, create simple foreground crops and masks, cache image features, and validate series boundaries by adjacent similarity.

In [ ]:
series = detect_series(manifest, config)
print("Series summary rows:", len(series["summary"]))
pprint(series["summary"][:5])

artifacts = compute_artifacts(series, config)
print("\nArtifact summary:")
pprint(artifacts["summary"])
print("Validated series rows:", len(artifacts["series_summary"]))

## 5. Extract Image Embeddings

The artifact step writes embeddings to the run cache. `feature_method="color_texture"` is the lightweight baseline; set `feature_method="dinov3"` after installing the DINOv3 model stack.

In [ ]:
feature_ready = [record for record in artifacts["records"] if record.get("feature_status") == "ready"]
print(f"Feature-ready images: {len(feature_ready)} / {len(artifacts['records'])}")
if feature_ready:
    first_vector = feature_ready[0]["feature_vector"]
    print(f"Feature method: {feature_ready[0].get('feature_method')}")
    print(f"Embedding dimension: {len(first_vector)}")
    print(f"First embedding cache: {feature_ready[0].get('feature_path')}")

## 6. Run Dimensionality Reduction

Create two-dimensional coordinates for quick visualization. PCA is used here because it is deterministic and available through NumPy.

In [ ]:
import csv
import numpy as np

reduced_records = []
if len(feature_ready) >= 2:
    matrix = np.asarray([record["feature_vector"] for record in feature_ready], dtype=np.float32)
    centered = matrix - matrix.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    coordinates = centered @ vt[:2].T
    for record, coordinate in zip(feature_ready, coordinates):
        reduced_records.append({
            "image_id": record.get("image_id"),
            "filename": record.get("filename"),
            "day_label": record.get("day_label"),
            "ground_truth_whale_id": record.get("ground_truth_whale_id"),
            "pca_x": float(coordinate[0]),
            "pca_y": float(coordinate[1]) if coordinates.shape[1] > 1 else 0.0,
        })

reduced_path = config.artifact_dir("embeddings") / "pca_coordinates.csv"
reduced_path.parent.mkdir(parents=True, exist_ok=True)
if reduced_records:
    with reduced_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=list(reduced_records[0]))
        writer.writeheader()
        writer.writerows(reduced_records)
print(f"Reduced coordinates: {len(reduced_records)} written to {reduced_path}")

## 7. Cluster Whale Images

Aggregate image embeddings into series representations, score same-day series pairs, and cluster connected components above the similarity threshold.

In [ ]:
matches = score_series_matches(artifacts, config)
clusters = cluster_day(matches, config)

print(f"Series representations: {len(matches['series'])}")
print(f"Pair scores: {len(matches['pair_scores'])}")
print(f"Predicted clusters: {len(clusters['summary'])}")
pprint(clusters["summary"][:10])

## 8. Evaluate Cluster Quality

Compute pairwise precision/recall/F1, adjusted Rand index, normalized mutual information, purity, and split/merge diagnostics when Lightroom labels are available.

In [ ]:
metrics = evaluate_clusters(clusters, ground_truth, config)
print("Overall metrics:")
for key, value in metrics.items():
    if key != "by_day":
        print(f"{key}: {value}")
print("\nDay metrics:")
pprint(metrics.get("by_day", []))

## 9. Visualize Clusters

Review representative series and predicted clusters with cached crops/previews, plus a PCA scatter when reduced coordinates exist.

In [ ]:
import matplotlib.pyplot as plt

plot_image_grid(artifacts, max_images=12)
plot_series_grid(artifacts, max_series=12)
plot_cluster_review(clusters, manifest, ground_truth, max_clusters=8)

if reduced_records:
    xs = [record["pca_x"] for record in reduced_records]
    ys = [record["pca_y"] for record in reduced_records]
    plt.figure(figsize=(6, 5))
    plt.scatter(xs, ys, s=18, alpha=0.75)
    plt.title("PCA view of cached embeddings")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()

## 10. Export Cluster Assignments and Artifacts

All major helpers write CSV/JSON artifacts as they run. This cell prints the files created for the current run so the output can be compared against `plan.md` milestones.

In [ ]:
artifact_files = sorted(path.relative_to(config.resolved_run_dir) for path in config.resolved_run_dir.rglob("*") if path.is_file())
print(f"Run artifacts in {config.resolved_run_dir}:")
for path in artifact_files:
    print(path)

progress_summary = {
    "target_day": config.target_day,
    "manifest_images": len(manifest["records"]),
    "lightroom_labeled_images": manifest["summary"].get("labeled_images"),
    "validated_series": len(artifacts["series_summary"]),
    "predicted_clusters": len(clusters["summary"]),
    "metrics_path": str(config.artifact_dir("reports") / "cluster_metrics.json"),
}
progress_path = config.artifact_dir("reports") / "progress_summary.json"
progress_path.write_text(json.dumps(progress_summary, indent=2, sort_keys=True), encoding="utf-8")
pprint(progress_summary)